# 作业5: 机器翻译

- 英译中

- <font color=darkred><b>***TODO***:</font></b>
    - <font color=darkred>训练一个简单的 RNN seq2seq模型，实现翻译功能</font>
    - <font color=darkred>改用transformer模型提升模型表现</font>
    - <font color=darkred>Apply Back-translation to furthur boost performance</font>

## 导入包

安装： 

- **editdistance**： 快速实现编辑距离（Levenshtein距离）。  
- **sacrebleu**： 计算bleu的库, 可以查看下[知乎: BLEU指标及评测脚本使用的一些误解](https://zhuanlan.zhihu.com/p/404381278)
- **sacremoses**: 使用Python实现了Moses的tokenizer, truecaser以及normalizer功能，使用起来比较方便[官方Github（有示例）](https://github.com/alvations/sacremoses)
- **sentencepiece**： 由谷歌将一些词-语言模型相关的论文进行复现，开发了一个开源工具——训练自己领域的sentencepiece模型，该模型可以代替预训练模型(BERT,XLNET)中词表的作用，可以参考[sentencepiece原理与实践](https://zhuanlan.zhihu.com/p/159200073)
- **wandb**: 是Weights & Biases的缩写，这款工具能够帮助跟踪你的机器学习项目。它能够自动记录模型训练过程中的超参数和输出指标，然后可视化和比较结果，并快速与同事共享结果。[官方文档：quickstart](https://docs.wandb.ai/v/zh-hans/quickstart)
- **fairseq**: 一个用PyTorch编写的序列建模工具包，它允许研究人员和开发人员训练用于翻译、摘要、语言建模和其他文本生成任务的自定义模型。[fairseq官方文档](https://fairseq.readthedocs.io/en/latest/)

fairseq 装了半天装不上，不用它了，用其他方法平替吧。

In [5]:
!pip install  editdistance  sacrebleu sacremoses sentencepiece wandb

    PyYAML (>=5.1.*)
            ~~~~~~^


In [6]:
import sys
import os
import pdb
import pprint
import logging
import random
import re

import numpy as np
# from tqdm.auto import tqdm 
from tqdm import tqdm # auto下载之后会显示不出来
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from argparse import Namespace
# from fairseq import utils
import matplotlib.pyplot as plt

## 一些功能函数

In [7]:
def all_seed(seed=6666):
    np.random.seed(seed)
    random.seed(seed)
    # CPU
    torch.manual_seed(seed)
    # GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
    # python全局
    os.environ['PYTHONHASHSEED'] = str(seed)
    # cudnn
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False
    print(f'Set env random_seed = {seed}')
    
    
all_seed(73)

Set env random_seed = 73


## 下载数据集

In [ ]:
data_dir = './DATA/rawdata'
dataset_name = 'ted2020'
urls = (
    "https://github.com/yuhsinchan/ML2022-HW5Dataset/releases/download/v1.0.2/ted2020.tgz",
    "https://github.com/yuhsinchan/ML2022-HW5Dataset/releases/download/v1.0.2/test.tgz",
)
file_names = (
    'ted2020.tgz', # train & dev
    'test.tgz', # test
)
prefix = Path(data_dir).absolute() / dataset_name

prefix.mkdir(parents=True, exist_ok=True)
for u, f in zip(urls, file_names):
    path = prefix / f
    print(f"download {u} to {path}")
    # 不存在则直接通过 weget进行下载
    if not path.exists():
        !wget {u} -O {path}
    if path.suffix == ".tgz":
        !tar -xvf {path} -C {prefix}
    elif path.suffix == ".zip":
        !unzip -o {path} -d {prefix}
!mv {prefix/'raw.en'} {prefix/'train_dev.raw.en'}
!mv {prefix/'raw.zh'} {prefix/'train_dev.raw.zh'}
!mv {prefix/'test/test.en'} {prefix/'test.raw.en'}
!mv {prefix/'test/test.zh'} {prefix/'test.raw.zh'}
!rm -rf {prefix/'test'}

download https://github.com/yuhsinchan/ML2022-HW5Dataset/releases/download/v1.0.2/ted2020.tgz to /Users/tomatoyuan/Sync/win/基础学习/leedl-self-learning/my_hw/HW5_Seq2Seq/DATA/rawdata/ted2020/ted2020.tgz
download https://github.com/yuhsinchan/ML2022-HW5Dataset/releases/download/v1.0.2/test.tgz to /Users/tomatoyuan/Sync/win/基础学习/leedl-self-learning/my_hw/HW5_Seq2Seq/DATA/rawdata/ted2020/test.tgz
